# Toutes les fonctions

In [ ]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from matplotlib import pyplot as plt
from wordcloud import WordCloud
from glob import glob
import os

# Stopwords

def fetch_stopwords():
    filepath = "My_stopwords.txt"
    all_stopwords = set(stopwords.words('english'))
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            raw_data = f.read()
        data_fetch = raw_data.split(",")
        mine_stopwords = [word.strip().strip('"').strip("'").lower() for word in data_fetch if word.strip()]
        all_stopwords.update(mine_stopwords)
        return all_stopwords
    except FileNotFoundError:
        print("Le fichier My_stopwords.txt est introuvable. Les stopwords par défaut seront utilisés.")
        return all_stopwords
    except Exception as e:
        print(f"Une erreur est survenue : {e}")
        return all_stopwords


def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)


#function to clean each book text
def preprocessing(book__path):

    try:
        with open(book__path, "r", encoding="utf-8", errors="ignore") as f:
            all_text = f.read()
    except FileNotFoundError:
        print(f"Fichier introuvable : {book__path}")
        return []

    # Supprimer le texte à la fin et retirer les lignes vides
    book_text = all_text.split("End of the Project")[0]
    lines = [line.strip() for line in book_text.split("\n") if line.strip()]

    #tokenisation
    tokens = []
    for line in lines:
        tokens.extend(word_tokenize(line))

    mine_stopwords = fetch_stopwords()

    clean_tokens = [w for w in tokens if w.isalpha() and len(w) > 1]
    clean_tokens = [w.lower() for w in clean_tokens if w.lower() not in mine_stopwords]

    # Lemmatisation
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in clean_tokens]

    lemmatized_words = [w for w in lemmatized_words if w not in mine_stopwords]

    return lemmatized_words

# Function to get frequencies
def book_freq(cleaned_data):
   return nltk.FreqDist(cleaned_data)


#function to generate the word cloud
def books_cloud(frequency, book__name):
    dir_path = "WordClouds"
    cloud_file = book__name + '.png'
    book_cloud = WordCloud(background_color="white", width=1000, height=500, colormap='rainbow', ).generate_from_frequencies(frequency)
    plt.figure(figsize = (12, 12))
    plt.imshow(book_cloud)
    plt.title(book__name)
    plt.axis("off")
    plt.show()
    book_cloud.to_file(f"{os.path.join(dir_path,cloud_file)}")


#function to create the bag of word
def book_freq_data(frequence_):
    # book_df = pd.DataFrame.from_dict(frequence_, orient='index', columns=['Frequency'])
    book_df = pd.DataFrame(frequence_.items(), columns=['word', 'frequency'])
    # book_df.index.name = 'Word'
    book_df = book_df.sort_values(by='frequency', ascending=False)
    return book_df

## LOOP FOR ALL BOOKS

In [ ]:
books_path = glob("../Data - NLP/*.txt")
books_path = sorted(books_path)
all_book_name = []
for book_path in books_path:
    book_name = book_path.split("/")[-1].split(".")[0]
    all_book_name.append(book_name)
    data = preprocessing(book_path)
    frequence = book_freq(data)
    bag_of_word = book_freq_data(frequence)
    books_cloud(frequence, book_name)
    print(bag_of_word)

In [ ]:
# html_file = "wordcloud.html"
#
# contenu_html = []
# h2_html = []
# for book__ in all_book_name:
#     contenu_html.append(f"""<h2>{book__}</h2> \n\n <img src="./WordClouds/{book__}.png" alt="Nuage de mots généré pour {book__}"/>""")
#     h2_html.append(f"""<h2>{book__}</h2>""")
#
# with open(html_file, "w", encoding="utf-8") as f:
#     f.write(str(contenu_html))



In [ ]:
# books_path = glob("../Data - NLP/*.txt")
# books_path = sorted(books_path)
# bags_of_words = {}
# for book_path in books_path:
#     book_name = book_path.split("/")[-1].split(".")[0]
#     data = preprocessing(book_path)
#     frequence = book_freq(data)
#     bag_of_word = book_freq_data(frequence)
#     bags = " ".join(bag_of_word['word'].tolist())
#     # books_cloud(frequence, book_name)
#     # print(bag_of_word)
#     bags_of_words[book_name] = bags
#
# bags_of_words
